# Regularization for Deep Learning — Notebook 2 of 3
## L¹ Regularization · Sparsity · The Priors Behind the Penalties

**Companion to Chapter 7, Stations 4–5.**  Notebook 1 showed L² *shrinking* weights. Swap the squared norm for the plain **sum of absolute values** and the behaviour changes qualitatively:

> L¹ does not merely shrink weights — it drives many of them to **exactly zero**, performing feature selection as a side effect. This is the **LASSO**.

### What you will do here
1. **Soft thresholding** — the shrinkage bench: L¹ vs L² on the same axes.
2. **Why zeros appear** — a constant push `α·sign(w)` finishes the job; `αw` never does.
3. **LASSO feature selection** — recover a sparse model buried in junk features.
4. **Implement ISTA yourself** — proximal gradient in ~10 lines.
5. **The Bayesian view** — Gaussian prior ⇒ L², Laplace prior ⇒ L¹, and the cusp that makes sparsity.
6. **Geometry** — why the diamond's corners land weights on the axes.
7. **Elastic Net** — have both.

Run top to bottom. **🔧 Try it** cells are editable; **✏️ Exercise** cells are yours to write.

## 0 · Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
plt.rcParams["figure.figsize"] = (7, 4.2)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

C_DATA = "#0F7C87"   # teal  — data term / w*
C_L2   = "#6C30D9"   # violet— L2
C_L1   = "#C81C6E"   # rose  — L1 / sparsity
C_WARN = "#B27412"   # amber
print("Ready.")

## 1 · Soft thresholding — the shrinkage bench

The L¹ penalty and its gradient:

$$\Omega(\theta)=\lVert w\rVert_1=\sum_i|w_i|\quad\Longrightarrow\quad \nabla_w\tilde J=\nabla_w J+\alpha\,\operatorname{sign}(w)$$

The penalty gradient `α·sign(w)` is a **constant** push toward zero — its size does *not* shrink as `w` shrinks. Under a diagonal-Hessian approximation the solution has a famous closed form, **soft thresholding**:

$$\tilde w_i=\operatorname{sign}(w^*_i)\,\max\!\Big\{\,|w^*_i|-\tfrac{\alpha}{H_{ii}},\;0\,\Big\}$$

Compare it to L²'s proportional shrink `w̃ = H/(H+α) · w*`. Plotting output weight `w̃` against unregularized `w*` makes the difference unmistakable.

In [ ]:
def soft_threshold(w_star, alpha, H):
    t = alpha / H                                  # dead-zone half-width
    return np.sign(w_star) * np.maximum(np.abs(w_star) - t, 0.0)

def l2_shrink(w_star, alpha, H):
    return (H / (H + alpha)) * w_star

def bench(alpha=0.6, H=1.0):
    ws = np.linspace(-2.5, 2.5, 400)
    plt.figure(figsize=(6.4, 6))
    plt.plot(ws, ws, color="gray", ls="--", lw=1, label="w̃ = w*  (no reg.)")
    plt.plot(ws, l2_shrink(ws, alpha, H),   color=C_L2, lw=2.2, label="L²  (proportional shrink)")
    plt.plot(ws, soft_threshold(ws, alpha, H), color=C_L1, lw=2.2, label="L¹  (soft threshold)")
    t = alpha / H
    plt.axvspan(-t, t, color=C_L1, alpha=0.10)
    plt.axhline(0, color="k", lw=0.6); plt.axvline(0, color="k", lw=0.6)
    plt.title(f"α={alpha:g}, H={H:g}   →   L¹ dead-zone ±{t:.2f},   L² slope {H/(H+alpha):.2f}")
    plt.xlabel("unregularized weight  w*"); plt.ylabel("output weight  w̃")
    plt.gca().set_aspect("equal"); plt.legend(loc="upper left"); plt.show()

bench()   # static default
print("Rose is FLAT and pinned to 0 inside the shaded dead-zone, then runs parallel to the diagonal.")
print("Violet passes through the origin with slope < 1 — shrunk, but essentially never exactly 0.")

**🔧 Try it — interactive.** Increase `α`: the L¹ dead-zone (shaded band) widens, swallowing more small weights to exactly zero, while L² just tilts its line flatter.

In [ ]:
try:
    from ipywidgets import interact, FloatSlider
    interact(bench,
             alpha=FloatSlider(min=0.0, max=2.0, step=0.05, value=0.6, description="α"),
             H=FloatSlider(min=0.3, max=3.0, step=0.1, value=1.0, description="H"));
except Exception as e:
    for a in [0.2, 0.6, 1.2]:
        bench(alpha=a)

### Why sparsity, in one line

The push toward `0` is what differs:

* **L²** pushes with `α·w` — it *fades* as the weight shrinks, so it never quite finishes.
* **L¹** pushes with `α·sign(w)` — a *constant* that keeps shoving until the weight is pinned at `0`.

Let's watch a single small weight under each rule.

In [ ]:
def descend(rule, w0=0.15, alpha=0.5, eps=0.05, steps=60):
    w, hist = w0, [w0]
    for _ in range(steps):
        if rule == "L2":
            w = w - eps * (alpha * w)                 # gradient step, data grad = 0
        else:  # L1 via proximal (soft-threshold) step
            w = w - eps * 0.0                          # (no data term)
            t = eps * alpha
            w = np.sign(w) * max(abs(w) - t, 0.0)
        hist.append(w)
    return np.array(hist)

plt.figure()
plt.plot(descend("L2"), color=C_L2, lw=2, label="L²: αw push  (asymptotes above 0)")
plt.plot(descend("L1"), color=C_L1, lw=2, label="L¹: constant push  (hits exactly 0)")
plt.axhline(0, color="k", lw=0.6)
plt.xlabel("step"); plt.ylabel("weight value")
plt.title("Same starting weight, two penalties — only L¹ reaches exactly zero")
plt.legend(); plt.show()

## 2 · LASSO in action — feature selection

The real payoff: given many more features than you need, L¹ tells you **which ones matter** by zeroing the rest. We build a ground-truth linear model where **only 5 of 40 features are real**, bury it in noise, and compare Ordinary Least Squares, Ridge (L²) and Lasso (L¹).

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso

rng = np.random.default_rng(1)
n, d, k = 80, 40, 5                       # 80 samples, 40 features, only 5 nonzero
X = rng.normal(size=(n, d))
true_w = np.zeros(d)
true_w[rng.choice(d, k, replace=False)] = rng.normal(3, 1, k)   # 5 real signals
y = X @ true_w + rng.normal(0, 0.5, n)

ols   = LinearRegression(fit_intercept=False).fit(X, y).coef_
ridge = Ridge(alpha=1.0, fit_intercept=False).fit(X, y).coef_
lasso = Lasso(alpha=0.1, fit_intercept=False, max_iter=10000).fit(X, y).coef_

fig, ax = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
for a, w, name, col in zip(ax, [ols, ridge, lasso],
                           ["OLS (no reg.)", "Ridge  L²", "Lasso  L¹"],
                           [C_WARN, C_L2, C_L1]):
    a.stem(true_w, linefmt="gray", markerfmt="D", basefmt=" ", label="true")
    a.stem(w, linefmt=col, markerfmt="o", basefmt=" ", label=name)
    a.set_ylabel("coef"); a.legend(loc="upper right")
    a.set_title(f"{name} — exact zeros: {np.sum(np.abs(w) < 1e-6)} / {d}")
ax[-1].set_xlabel("feature index")
plt.tight_layout(); plt.show()

print("Lasso zeros out the junk features; OLS and Ridge keep small nonzero weight on all 40.")

**🔧 Try it — the LASSO path.** As `α` rises, coefficients drop to exactly zero one after another — a built-in ranking of feature importance.

In [ ]:
alphas = np.logspace(-2, 0.6, 60)
paths = np.array([Lasso(alpha=a, fit_intercept=False, max_iter=10000).fit(X, y).coef_
                  for a in alphas])
active = (np.abs(paths) > 1e-6).sum(axis=1)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for j in range(d):
    col = C_L1 if true_w[j] != 0 else "lightgray"
    ax[0].plot(alphas, paths[:, j], color=col, lw=1.6 if true_w[j] != 0 else 0.8)
ax[0].set_xscale("log"); ax[0].set_xlabel("α"); ax[0].set_ylabel("coefficient")
ax[0].set_title("LASSO paths (rose = truly nonzero, grey = junk)")
ax[1].plot(alphas, active, color=C_L1, marker="o", ms=3)
ax[1].axhline(k, color="gray", ls="--", label=f"true count = {k}")
ax[1].set_xscale("log"); ax[1].set_xlabel("α"); ax[1].set_ylabel("# nonzero coefficients")
ax[1].set_title("Sparsity increases with α"); ax[1].legend()
plt.tight_layout(); plt.show()

**✏️ Exercise 2.** Use validation to pick `α`. Split `(X, y)` into train/validation, fit Lasso across the `alphas` grid, and plot validation MSE vs `α`. Does the `α` that minimizes validation error also recover close to the true 5 nonzero features? (Hint: `from sklearn.model_selection import train_test_split`, `from sklearn.metrics import mean_squared_error`.)

In [ ]:
# ✏️ Your code here.


## 3 · Implement ISTA yourself

The proximal-gradient / **Iterative Shrinkage-Thresholding Algorithm** solves L¹ problems by alternating a plain gradient step on the data term with a soft-threshold:

```
w ← w − ε·∇(data loss)          # ordinary gradient step
w ← sign(w)·max(|w| − εα, 0)    # soft-threshold toward 0
```

Fill in the two lines and recover a sparse signal from noisy linear measurements.

In [ ]:
def soft(w, t):
    return np.sign(w) * np.maximum(np.abs(w) - t, 0.0)

def ista(X, y, alpha=0.1, eps=None, steps=400):
    n, d = X.shape
    if eps is None:
        # gradient of (1/2n)‖Xw−y‖² has Lipschitz constant ‖X‖₂²/n, so 1/L = n/‖X‖₂²
        eps = n / np.linalg.norm(X, 2) ** 2        # safe step size (1/Lipschitz)
    w = np.zeros(d)
    obj = []
    for _ in range(steps):
        grad = X.T @ (X @ w - y) / n               # ∇ of ½‖Xw−y‖²/n
        w = w - eps * grad                          # (1) gradient step
        w = soft(w, eps * alpha)                    # (2) soft-threshold
        obj.append(0.5 * np.mean((X @ w - y) ** 2) + alpha * np.abs(w).sum())
    return w, obj

w_hat, obj = ista(X, y, alpha=0.1, steps=600)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].stem(true_w, linefmt="gray", markerfmt="D", basefmt=" ", label="true")
ax[0].stem(w_hat, linefmt=C_L1, markerfmt="o", basefmt=" ", label="ISTA")
ax[0].set_title(f"ISTA recovery — nonzeros: {np.sum(np.abs(w_hat) > 1e-6)}"); ax[0].legend()
ax[1].plot(obj, color=C_L1); ax[1].set_title("Objective (data + α·‖w‖₁) decreasing")
ax[1].set_xlabel("iteration"); ax[1].set_ylabel("objective")
plt.tight_layout(); plt.show()

# Compare to scikit-learn's Lasso
sk = Lasso(alpha=0.1, fit_intercept=False, max_iter=10000).fit(X, y).coef_
print("max |ISTA − sklearn Lasso|:", np.max(np.abs(w_hat - sk)).round(3),
      " (small = your implementation matches the library)")

## 4 · The Bayesian view — Gaussian vs Laplace priors

Both penalties fall out of the same idea: put a **prior** on the weights ("small is more probable"), then do **MAP** estimation.

$$\text{MAP: } \max_w\;\log p(y\mid X,w)+\log p(w),\qquad \log p(w)=-\alpha\,\Omega(w)+\text{const}$$

* A **Gaussian** prior `N(0, 1/α)` gives `−log p ∝ ½w²` → **L²**.
* A **Laplace** prior gives `−log p ∝ |w|` → **L¹**.

The shapes explain everything: the Laplace has a sharp **cusp at 0**, so the MAP estimate is happy to park a weight exactly at the peak. The Gaussian is smooth through 0, so it never prefers exact zeros.

In [ ]:
w = np.linspace(-4, 4, 500)
b = 1.0                      # Laplace scale
sig = 1.0                    # Gaussian std
gauss   = np.exp(-w**2 / (2 * sig**2)) / (sig * np.sqrt(2 * np.pi))
laplace = np.exp(-np.abs(w) / b) / (2 * b)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot(w, gauss,   color=C_L2, lw=2.2, label="Gaussian prior  → L²")
ax[0].plot(w, laplace, color=C_L1, lw=2.2, label="Laplace prior  → L¹")
ax[0].set_title("Prior densities  p(w)"); ax[0].legend(); ax[0].set_xlabel("w")
# Negative log priors ARE the penalties (up to constants):
ax[1].plot(w, 0.5 * w**2, color=C_L2, lw=2.2, label="−log Gaussian ∝ ½w²  (L²)")
ax[1].plot(w, np.abs(w),  color=C_L1, lw=2.2, label="−log Laplace ∝ |w|   (L¹)")
ax[1].set_title("Negative log-prior = the penalty Ω(w)"); ax[1].legend(); ax[1].set_xlabel("w")
plt.tight_layout(); plt.show()

print("The cusp of |w| at 0 (rose, right) is the exact origin of the soft-threshold dead zone.")

## 5 · Geometry — why the corner is the point

Every norm penalty is also a **constraint region** the weights must stay inside (we'll formalize this in Notebook 3). The *shape* of that region is what drives sparsity:

* **L²** region is a **circle/sphere** — smooth, no corners.
* **L¹** region is a **diamond** (cross-polytope) — corners sit *on the axes*.

A loss contour growing outward usually first touches the diamond **at a corner**, and a corner means one coordinate is exactly zero. The sphere has no corners, so L² almost never lands a weight on an axis.

In [ ]:
# Elliptical loss contours centred at w* with an L1 diamond and L2 circle of matched "budget".
w_star = np.array([1.6, 0.6])
Hd = np.array([1.0, 3.0])           # anisotropic curvature -> tilted-looking contours

def loss(P):
    D = P - w_star
    return 0.5 * (Hd[0] * D[..., 0]**2 + Hd[1] * D[..., 1]**2)

g = np.linspace(-2, 2.4, 400)
A, B = np.meshgrid(g, g)
Z = loss(np.stack([A, B], -1))

t = np.linspace(0, 2*np.pi, 400)
r = 1.0
circle = np.stack([r*np.cos(t), r*np.sin(t)], -1)                      # L2 ball
s = np.linspace(-1, 1, 200)
diamond = np.array([[ r, 0], [0,  r], [-r, 0], [0, -r], [r, 0]])       # L1 ball corners

fig, ax = plt.subplots(1, 2, figsize=(12, 5.6))
for a, region, name, col in zip(ax, [circle, diamond], ["L² ball (circle)", "L¹ ball (diamond)"], [C_L2, C_L1]):
    a.contour(A, B, Z, levels=18, colors=C_DATA, alpha=0.45, linewidths=0.7)
    a.plot(region[:, 0], region[:, 1], color=col, lw=2.2)
    a.scatter(*w_star, color=C_DATA, s=60, zorder=5, label="w*")
    a.axhline(0, color="k", lw=0.5); a.axvline(0, color="k", lw=0.5)
    a.set_aspect("equal"); a.set_title(name); a.legend(loc="upper right")
    a.set_xlim(-2, 2.4); a.set_ylim(-2, 2.4)
plt.suptitle("Contours meet the L¹ diamond at a corner (a weight = 0); the L² circle almost never on-axis")
plt.tight_layout(); plt.show()

## 6 · Elastic Net — have both

In practice you rarely have to choose. **Elastic Net** mixes L¹ and L², giving sparsity *and* stability. It shines when features come in correlated groups: pure LASSO keeps one member at random and drops the rest, while the L² part keeps the group together.

In [ ]:
from sklearn.linear_model import ElasticNet

ratios = [0.0, 0.25, 0.5, 0.75, 1.0]     # 0 = pure Ridge-like, 1 = pure Lasso
counts = []
for lr in ratios:
    # l1_ratio=0 is invalid for ElasticNet; nudge it slightly
    en = ElasticNet(alpha=0.1, l1_ratio=max(lr, 1e-3),
                    fit_intercept=False, max_iter=10000).fit(X, y)
    counts.append(np.sum(np.abs(en.coef_) > 1e-6))

plt.figure()
plt.plot(ratios, counts, color=C_L1, marker="o")
plt.xlabel("l1_ratio  (0 = ridge-like → 1 = lasso-like)")
plt.ylabel("# nonzero coefficients")
plt.title("Elastic Net slides between dense (L²) and sparse (L¹)")
plt.axhline(k, color="gray", ls="--", label=f"true nonzero = {k}"); plt.legend(); plt.show()

print("More L¹ (higher l1_ratio) → sparser solution; more L² → denser but more stable.")

## Key takeaways

1. **L¹ is soft thresholding.** `w̃ᵢ = sign(w*ᵢ)·max(|w*ᵢ| − α/Hᵢᵢ, 0)`. A constant push `α·sign(w)` pins small weights at exactly `0`.
2. **Sparsity = feature selection.** LASSO zeros out irrelevant features and ranks the rest; the path shows which survive longest.
3. **ISTA** = gradient step + soft-threshold; ten lines reproduce scikit-learn's Lasso.
4. **Priors behind the penalties.** L² = MAP with a Gaussian prior; L¹ = MAP with a Laplace prior. The Laplace cusp at `0` is where sparsity comes from.
5. **Geometry.** The L¹ diamond's corners lie on the axes, so contours touch there — one coordinate exactly zero. The L² circle has no corners.
6. **Elastic Net** blends both for correlated features.

**Next → Notebook 3:** every penalty is secretly a **constraint**. We'll make that precise (the Lagrangian), implement **reprojection / max-norm**, and see how a dab of regularization rescues **under-constrained** problems where the unregularized answer runs off to infinity.